# VCFingerprinting: Attack and Defense Workflow

This notebook combines the VCFP Attack (Voice Command Fingerprint Attack) and BuFLO Defense workflows into a single sequential pipeline.

## Workflow Overview:
1. **Data Preprocessing**: Convert PCAP files to CSV format using TrafficLoader
2. **Burst Separation**: Separate traffic into individual voice command bursts
3. **BuFLO Defense**: Apply BuFLO (Bursting Flow) countermeasure to traffic data
4. **VCFP Attack**: Classify/identify voice commands from network traffic
5. **Comparison Analysis**: Compare results between defended and undefended traffic

## Section 1: Setup and Imports

In [2]:
# Import necessary libraries
import os
import sys
import csv
import numpy as np
import pandas as pd
import random
from pathlib import Path
from collections import defaultdict
import math
import re

# For PCAP processing - using service modules from the repository
from services.pcap_to_csv import TrafficLoader
from services.seperate_bursts import separate_bursts

# For machine learning
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import classification_report

# Set working directory
WORK_DIR = '/home/exorcist1770/Desktop/Chenggang wang/Task_2/task_2_data/voice_command_fingerprinting'
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

Working directory: /home/exorcist1770/Desktop/Chenggang wang/Task_2/task_2_data/voice_command_fingerprinting


## Section 2: Data Preprocessing - PCAP to CSV Conversion

This section converts PCAP network capture files to CSV format using the repository's service modules:
- **TrafficLoader**: Converts PCAP to trace CSV
- **separate_bursts**: Separates trace into individual voice command bursts

In [3]:
# Data directories - Updated to match repository structure
CAPTURED_FILES_DIR = 'data/captured_files/'
TRACE_CSV_DIR = 'data/trace_csv_files/'
SEPARATED_BURSTS_DIR = 'data/seperated_bursts_files/'
BUFLO_OUTPUT_DIR = 'data/buflo/'

# Create directories if they don't exist
os.makedirs(TRACE_CSV_DIR, exist_ok=True)
os.makedirs(SEPARATED_BURSTS_DIR, exist_ok=True)
os.makedirs(BUFLO_OUTPUT_DIR, exist_ok=True)

# Example: List available PCAP files
print("Available PCAP files:")
if os.path.exists(CAPTURED_FILES_DIR):
    for f in os.listdir(CAPTURED_FILES_DIR):
        if f.endswith('.pcap'):
            print(f"  {os.path.join(CAPTURED_FILES_DIR, f)}")
else:
    print(f"  Directory not found: {CAPTURED_FILES_DIR}")

Available PCAP files:
  data/captured_files/how_deep_is_the_indian_ocean_5_30s.pcap


In [4]:
# Function to process PCAP file to burst-separated CSV
def process_pcap_to_bursts(pcap_filename, target_ip, gap_threshold=1.0):
    """
    Process a PCAP file to burst-separated CSV format.
    
    Args:
        pcap_filename: Name of the PCAP file
        target_ip: IP address of the target device (e.g., Amazon Echo)
        gap_threshold: Time gap threshold for burst separation (default: 1.0 seconds)
    
    Returns:
        Path to the burst-separated CSV file
    """
    # Construct full path
    pcap_path = os.path.join(CAPTURED_FILES_DIR, pcap_filename)
    
    if not os.path.exists(pcap_path):
        print(f"PCAP file not found: {pcap_path}")
        return None
    
    # Get base name without extension
    base_name = os.path.splitext(pcap_filename)[0]
    
    # Define output paths
    trace_csv = os.path.join(TRACE_CSV_DIR, f"{base_name}.csv")
    bursts_csv = os.path.join(SEPARATED_BURSTS_DIR, f"{base_name}_bursts.csv")
    
    # Step 1: Convert PCAP to trace CSV using TrafficLoader
    print(f"\nStep 1: Converting PCAP to trace CSV...")
    loader = TrafficLoader(target_ip)
    df_trace = loader.pcap_to_csv(pcap_path, trace_csv)
    
    if df_trace is None:
        print("Failed to convert PCAP to CSV")
        return None
    
    # Step 2: Separate bursts using separate_bursts function
    print(f"\nStep 2: Separating bursts...")
    df_bursts = separate_bursts(trace_csv, gap_threshold=gap_threshold)
    
    # Print burst statistics
    burst_counts = df_bursts['burst_id'].value_counts().sort_index()
    print(f"Found {len(burst_counts)} valid bursts")
    for burst_id, count in burst_counts.items():
        print(f"  Burst {burst_id}: {count} packets")
    
    # Save to CSV
    df_bursts.to_csv(bursts_csv, index=False)
    print(f"\nSaved bursts to: {bursts_csv}")
    
    return bursts_csv


# Example: Process a sample PCAP file
# Uncomment to run:
process_pcap_to_bursts('how_deep_is_the_indian_ocean_5_30s.pcap', '192.168.86.40')


Step 1: Converting PCAP to trace CSV...
Starting to Processing data/captured_files/how_deep_is_the_indian_ocean_5_30s.pcap for IP: 192.168.86.40
Successfully saved 2268 packets to data/trace_csv_files/how_deep_is_the_indian_ocean_5_30s.csv
Trace duration: 180.0997 seconds

Step 2: Separating bursts...
Loading data/trace_csv_files/how_deep_is_the_indian_ocean_5_30s.csv...
Found 10 valid voice commands.
Found 10 valid bursts
  Burst 3: 406 packets
  Burst 4: 43 packets
  Burst 6: 373 packets
  Burst 7: 54 packets
  Burst 10: 401 packets
  Burst 11: 36 packets
  Burst 14: 409 packets
  Burst 15: 45 packets
  Burst 17: 429 packets
  Burst 18: 43 packets

Saved bursts to: data/seperated_bursts_files/how_deep_is_the_indian_ocean_5_30s_bursts.csv


'data/seperated_bursts_files/how_deep_is_the_indian_ocean_5_30s_bursts.csv'

## Section 3: BuFLO Defense Implementation

BuFLO (Bursting Flow) is a defensive mechanism that:
- Sends fixed-length packets at regular intervals
- Pads small packets to a fixed size
- Chops large packets into fixed-size chunks
- Adds dummy packets to maintain minimum transmission time

This obfuscates the original traffic pattern making it harder to identify voice commands.

In [5]:
# Updated CSV loading for new format (timestamp, length, direction, burst_id, etc.)
def load_burst_csv(csv_path):
    """
    Load burst CSV data into a structured format.
    
    Args:
        csv_path: Path to the burst CSV file
    
    Returns:
        DataFrame with burst data
    """
    df = pd.read_csv(csv_path)
    return df


def apply_buflo_to_burst(df_burst, d=600, f=50, t=20):
    """
    Apply BuFLO countermeasure to a single burst.
    
    Args:
        df_burst: DataFrame with burst data (columns: timestamp, length, direction, burst_id)
        d: Fixed packet size (default: 600 bytes)
        f: Frequency of packet transmission (packets per second)
        t: Minimum transmission time (in seconds)
    
    Returns:
        DataFrame with BuFLO-applied traffic
    """
    if len(df_burst) == 0:
        return pd.DataFrame()
    
    buflo_data = []
    start_t = df_burst['timestamp'].min()
    end_time = df_burst['timestamp'].max()
    
    index = 0
    total_packet = int(t * f)  # Minimum packets needed
    total_overhead = 0
    
    # Process each packet in the burst
    for _, row in df_burst.iterrows():
        size = row['length']
        direction = row['direction']
        
        if size <= d:
            # Pad small packets
            overhead = d - size
            total_overhead += overhead
            new_p = {
                'index': index,
                'timestamp': round(start_t + index * (1 / f), 6),
                'length': d,
                'direction': direction,
                'overhead': overhead,
                'status': 'padded'
            }
            buflo_data.append(new_p)
            index += 1
        else:
            # Chop large packets
            remaining = size
            while remaining > 0:
                chunk_size = min(d, remaining)
                overhead = d - chunk_size if remaining <= d else 0
                total_overhead += overhead
                new_p = {
                    'index': index,
                    'timestamp': round(start_t + index * (1 / f), 6),
                    'length': chunk_size,
                    'direction': direction,
                    'overhead': overhead,
                    'status': 'chopped'
                }
                buflo_data.append(new_p)
                remaining -= chunk_size
                index += 1
    
    # Add dummy packets to meet minimum time requirement
    if index < total_packet:
        for i in range(index, total_packet):
            seed = -1 + 2 * random.random()
            direction = float(np.sign(seed))
            dummy_packet = {
                'index': i,
                'timestamp': round(start_t + (i + 1) * (1 / f), 6),
                'length': d,
                'direction': direction,
                'overhead': d,
                'status': 'dummy'
            }
            total_overhead += d
            buflo_data.append(dummy_packet)
    
    # Create DataFrame
    df_buflo = pd.DataFrame(buflo_data)
    
    time_delay = df_buflo['timestamp'].max() - end_time if len(df_buflo) > 0 else 0
    
    print(f"  Time delay: {time_delay:.2f}s")
    print(f"  Total overhead: {total_overhead} bytes")
    print(f"  Original packets: {len(df_burst)}, BuFLO packets: {len(df_buflo)}")
    
    return df_buflo


def apply_buflo_to_file(csv_path, d=600, f=50, t=20, output_dir=BUFLO_OUTPUT_DIR):
    """
    Apply BuFLO to all bursts in a CSV file.
    
    Args:
        csv_path: Path to the burst CSV file
        d: Fixed packet size
        f: Frequency
        t: Minimum transmission time
        output_dir: Output directory
    
    Returns:
        Path to the defended CSV file
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Load the burst CSV
    df = load_burst_csv(csv_path)
    
    if len(df) == 0:
        print("Error: No data in CSV file")
        return None
    
    # Get base name
    base_name = Path(csv_path).stem
    
    # Get unique burst IDs
    burst_ids = df['burst_id'].unique()
    
    all_buflo_data = []
    
    print(f"Applying BuFLO to {len(burst_ids)} bursts...")
    
    for burst_id in sorted(burst_ids):
        df_burst = df[df['burst_id'] == burst_id].copy()
        df_buflo = apply_buflo_to_burst(df_burst, d, f, t)
        df_buflo['burst_id'] = burst_id
        all_buflo_data.append(df_buflo)
    
    # Combine all bursts
    df_combined = pd.concat(all_buflo_data, ignore_index=True)
    
    # Save to CSV
    output_path = os.path.join(output_dir, f"{base_name}_buflo.csv")
    df_combined.to_csv(output_path, index=False)
    
    print(f"BuFLO applied: {base_name}")
    print(f"  Output: {output_path}")
    
    return output_path


print("BuFLO defense functions defined successfully!")

BuFLO defense functions defined successfully!


## Section 4: VCFP Attack Implementation

The VCFP Attack uses machine learning to identify voice commands from network traffic patterns.

This section implements multiple classification methods:
- **Bayes**: Naive Bayes classifier
- **SVM**: Support Vector Machine
- **Jaccard**: Jaccard similarity-based classification
- **VNG++**: Variable N-gram based classification

In [6]:
# VCFP Attack - Feature Extraction and Classification
# Updated to work with new CSV format (length instead of size, timestamp instead of time)

def compute_bayes_feature(df, interval=50):
    """
    Compute features for Bayes classifier using packet size histograms.
    
    Args:
        df: DataFrame with traffic data (must have 'length' column)
        interval: Bucket size for histogram
    
    Returns:
        Feature vector
    """
    # Create size histogram
    start, end, step = -1500, 1501, interval
    ranges = list(range(start, end, step))
    features = [0] * len(ranges)
    
    for size in df['length']:
        idx = int((size - start) / step)
        if 0 <= idx < len(features):
            features[idx] += 1
    
    return features


def compute_vngpp_feature(df, interval=5000):
    """
    Compute VNG++ features using cumulative packet sizes.
    
    Args:
        df: DataFrame with traffic data (must have 'length' column)
        interval: Bucket size for cumulative sum
    
    Returns:
        Feature vector
    """
    # Compute cumulative sum of sizes
    cumsum = df['length'].cumsum().values
    
    # Create histogram
    start, end, step = -400000, 400001, interval
    ranges = list(range(start, end, step))
    features = [0] * len(ranges)
    
    for val in cumsum:
        idx = int((val - start) / step)
        if 0 <= idx < len(features):
            features[idx] += 1
    
    return features


def compute_jaccard_feature(df):
    """
    Compute Jaccard similarity features.
    
    Args:
        df: DataFrame with traffic data (must have 'length' column)
    
    Returns:
        Set of packet sizes
    """
    return set(df['length'].values)


def get_label_from_filename(csv_path):
    """
    Extract label (query name) from CSV filename.
    
    Args:
        csv_path: Path to CSV file
    
    Returns:
        Label string
    """
    filename = os.path.basename(csv_path)
    # Extract query name from filename pattern like: name_bursts.csv or name_1.csv
    # For burst files: remove _bursts suffix
    # For numbered files: remove _number suffix
    
    # Pattern: everything before _bursts or _
    match = re.match(r'^(.+?)(?:_bursts|_\d+)?\.csv$', filename)
    if match:
        return match.group(1)
    return filename[:-4]  # Remove .csv


def load_dataset_from_bursts(csv_dir, feature_method='bayes', interval=50):
    """
    Load dataset from burst CSV directory.
    
    Args:
        csv_dir: Directory containing burst CSV files
        feature_method: 'bayes', 'vngpp', or 'jaccard'
        interval: Interval for feature extraction
    
    Returns:
        X: Feature matrix
        y: Labels
        label_map: Mapping of labels to indices
    """
    X = []
    y = []
    label_map = defaultdict(int)
    label_count = 0
    
    for filename in os.listdir(csv_dir):
        if not filename.endswith('.csv'):
            continue
        
        csv_path = os.path.join(csv_dir, filename)
        
        # Load the burst CSV
        df = load_burst_csv(csv_path)
        
        if len(df) == 0:
            continue
        
        # Get label from filename
        label = get_label_from_filename(csv_path)
        
        # Get label index
        if label not in label_map:
            label_map[label] = label_count
            label_count += 1
        
        # Compute features based on method
        if feature_method == 'bayes':
            features = compute_bayes_feature(df, interval)
        elif feature_method == 'vngpp':
            features = compute_vngpp_feature(df, interval)
        elif feature_method == 'jaccard':
            features = compute_jaccard_feature(df)
        else:
            raise ValueError(f"Unknown feature method: {feature_method}")
        
        X.append(features)
        y.append(label_map[label])
    
    # Convert to numpy arrays
    X = np.array(X)
    y = np.array(y)
    
    print(f"Loaded {len(X)} samples with {len(label_map)} classes")
    print(f"Feature dimension: {X.shape[1] if len(X) > 0 else 0}")
    
    return X, y, dict(label_map)


print("VCFP Attack feature extraction functions defined successfully!")

VCFP Attack feature extraction functions defined successfully!


In [7]:
# Classification functions

def train_bayes(X_train, y_train):
    """Train Naive Bayes classifier."""
    clf = GaussianNB()
    clf.fit(X_train, y_train)
    return clf


def train_svm(X_train, y_train):
    """Train SVM classifier."""
    clf = svm.SVC()
    clf.fit(X_train, y_train)
    return clf


def jaccard_similarity(set1, set2):
    """Compute Jaccard similarity between two sets."""
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0


def train_jaccard(X_train, y_train):
    """Train Jaccard classifier (stores training sets)."""
    train_sets = {}
    for i, label in enumerate(y_train):
        if label not in train_sets:
            train_sets[label] = []
        train_sets[label].append(X_train[i])
    return train_sets


def predict_jaccard(clf, X_test):
    """Predict using Jaccard similarity."""
    predictions = []
    for test_set in X_test:
        best_label = None
        best_sim = -1
        
        for label, train_sets in clf.items():
            for train_set in train_sets:
                sim = jaccard_similarity(test_set, train_set)
                if sim > best_sim:
                    best_sim = sim
                    best_label = label
        
        predictions.append(best_label)
    
    return np.array(predictions)


def evaluate_classifier(clf, X_test, y_test, method='standard'):
    """
    Evaluate classifier performance.
    
    Args:
        clf: Trained classifier
        X_test: Test features
        y_test: True labels
        method: 'standard' or 'jaccard'
    
    Returns:
        accuracy: Classification accuracy
        avg_rank: Average rank (for semantic distance)
    """
    if method == 'jaccard':
        y_pred = predict_jaccard(clf, X_test)
    else:
        y_pred = clf.predict(X_test)
    
    # Calculate accuracy
    correct = np.sum(y_pred == y_test)
    accuracy = correct / len(y_test)
    
    # Calculate average rank (for semantic distance)
    avg_rank = 0
    
    return accuracy, avg_rank


print("Classification functions defined successfully!")

Classification functions defined successfully!


## Section 5: Cross-Validation and Model Training

In [8]:
def n_fold_cross_validation(X, y, n_folds=5, method='bayes'):
    """
    Perform n-fold cross-validation.
    
    Args:
        X: Feature matrix
        y: Labels
        n_folds: Number of folds
        method: Classification method ('bayes', 'svm', 'jaccard')
    
    Returns:
        avg_accuracy: Average accuracy across folds
        avg_rank: Average rank across folds
        fold_accuracies: List of accuracies per fold
    """

    # Count samples per class and handle classes with insufficient samples
    from collections import Counter
    class_counts = Counter(y)
    min_samples = min(class_counts.values())
    
    if min_samples < n_folds:
        # Find classes with fewer samples than n_folds
        insufficient_classes = [cls for cls, count in class_counts.items() if count < n_folds]
        print(f"WARNING: The following classes have fewer than {n_folds} samples: {insufficient_classes}")
        print(f"Filtering out these classes before cross-validation...")
        
        # Create mask to filter out insufficient classes
        mask = np.array([y_i not in insufficient_classes for y_i in y])
        X_filtered = X[mask]
        y_filtered = y[mask]
        
        # Update class counts after filtering
        new_class_counts = Counter(y_filtered)
        new_min_samples = min(new_class_counts.values())
        new_n_folds = min(n_folds, new_min_samples)
        
        print(f"After filtering: {len(X_filtered)} samples, {len(new_class_counts)} classes")
        print(f"Adjusted n_folds from {n_folds} to {new_n_folds}")
        
        X = X_filtered
        y = y_filtered
        n_folds = new_n_folds

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_accuracies = []
    fold_ranks = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Train classifier
        if method == 'bayes':
            clf = train_bayes(X_train, y_train)
        elif method == 'svm':
            clf = train_svm(X_train, y_train)
        elif method == 'jaccard':
            clf = train_jaccard(X_train, y_train)
        else:
            raise ValueError(f"Unknown method: {method}")
        
        # Evaluate
        accuracy, rank = evaluate_classifier(clf, X_test, y_test, method)
        fold_accuracies.append(accuracy)
        fold_ranks.append(rank)
        
        print(f"Fold {fold+1}: Accuracy = {accuracy:.4f}")
    
    avg_accuracy = np.mean(fold_accuracies)
    avg_rank = np.mean(fold_ranks)
    
    print(f"\nAverage Accuracy: {avg_accuracy:.4f}")
    print(f"Average Rank: {avg_rank:.4f}")
    
    return avg_accuracy, avg_rank, fold_accuracies


print("Cross-validation function defined successfully!")

Cross-validation function defined successfully!


## Section 6: Complete Workflow - Run Attack and Defense

In [9]:
def run_complete_workflow(data_dir, buflo_params=None, attack_method='bayes', n_folds=5):
    """
    Run the complete attack-defense workflow.
    
    Args:
        data_dir: Directory containing burst CSV files
        buflo_params: Dictionary with BuFLO parameters (d, f, t). If None, skip defense
        attack_method: Classification method
        n_folds: Number of cross-validation folds
    
    Returns:
        Dictionary with results
    """
    results = {}
    
    # Step 1: Load original data
    print("=" * 60)
    print("Step 1: Loading original (undefended) data...")
    print("=" * 60)
    
    X_orig, y_orig, label_map = load_dataset_from_bursts(data_dir, feature_method=attack_method)
    
    if len(X_orig) == 0:
        print("Error: No data loaded!")
        return None
    
    # Step 2: Run attack on original data
    print("\n" + "=" * 60)
    print("Step 2: Running attack on undefended traffic...")
    print("=" * 60)
    
    avg_acc_orig, avg_rank_orig, _ = n_fold_cross_validation(
        X_orig, y_orig, n_folds=n_folds, method=attack_method
    )
    
    results['undefended'] = {
        'accuracy': avg_acc_orig,
        'avg_rank': avg_rank_orig,
        'n_samples': len(X_orig),
        'n_classes': len(label_map)
    }
    
    # Step 3: Apply BuFLO defense if params provided
    if buflo_params:
        print("\n" + "=" * 60)
        print("Step 3: Applying BuFLO defense...")
        print("=" * 60)
        
        # Apply BuFLO to all files in the directory
        defended_files = []
        for filename in os.listdir(data_dir):
            if filename.endswith('.csv'):
                csv_path = os.path.join(data_dir, filename)
                output_path = apply_buflo_to_file(csv_path, **buflo_params)
                if output_path:
                    defended_files.append(output_path)
        
        # Step 4: Run attack on defended data
        print("\n" + "=" * 60)
        print("Step 4: Running attack on defended traffic...")
        print("=" * 60)
        
        X_def, y_def, _ = load_dataset_from_bursts(BUFLO_OUTPUT_DIR, feature_method=attack_method)
        
        if len(X_def) > 0:
            avg_acc_def, avg_rank_def, _ = n_fold_cross_validation(
                X_def, y_def, n_folds=n_folds, method=attack_method
            )
            
            results['defended'] = {
                'accuracy': avg_acc_def,
                'avg_rank': avg_rank_def,
                'n_samples': len(X_def),
                'params': buflo_params
            }
    
    return results


print("Complete workflow function defined successfully!")

Complete workflow function defined successfully!


## Section 7: Comparison Analysis

In [14]:
def compare_results(results):
    """
    Compare and visualize results between defended and undefended traffic.
    
    Args:
        results: Dictionary with 'undefended' and 'defended' results
    """
    print("\n" + "=" * 60)
    print("COMPARISON ANALYSIS: DEFENDED vs UNDEFENDED TRAFFIC")
    print("=" * 60)
    
    if 'undefended' in results:
        print(f"\n--- Undefended Traffic ---")
        print(f"  Accuracy: {results['undefended']['accuracy']:.4f}")
        print(f"  Samples: {results['undefended']['n_samples']}")
        print(f"  Classes: {results['undefended']['n_classes']}")
    
    if 'defended' in results:
        print(f"\n--- Defended Traffic (BuFLO) ---")
        print(f"  Accuracy: {results['defended']['accuracy']:.4f}")
        print(f"  Samples: {results['defended']['n_samples']}")
        print(f"  BuFLO Params: {results['defended']['params']}")
        
        # Calculate improvement
        acc_diff = results['undefended']['accuracy'] - results['defended']['accuracy']
        print(f"\n--- Impact of Defense ---")
        print(f"  Accuracy Drop: {acc_diff:.4f} ({acc_diff*100:.2f}%)")
        print(f"  Defense Effectiveness: {'HIGH' if acc_diff > 0.2 else 'MEDIUM' if acc_diff > 0.1 else 'LOW'}")
    
    print("\n" + "=" * 60)

## Section 8: Example Usage

Below is an example of how to use the complete workflow.

In [15]:
# Check what data files are available
print("Checking available data files...")
print(f"\nCaptured files: {CAPTURED_FILES_DIR}")
if os.path.exists(CAPTURED_FILES_DIR):
    pcap_files = [f for f in os.listdir(CAPTURED_FILES_DIR) if f.endswith('.pcap')]
    print(f"  Found {len(pcap_files)} PCAP files: {pcap_files}")

print(f"\nTrace CSV files: {TRACE_CSV_DIR}")
if os.path.exists(TRACE_CSV_DIR):
    trace_files = [f for f in os.listdir(TRACE_CSV_DIR) if f.endswith('.csv')]
    print(f"  Found {len(trace_files)} trace files")

print(f"\nSeparated bursts files: {SEPARATED_BURSTS_DIR}")
if os.path.exists(SEPARATED_BURSTS_DIR):
    burst_files = [f for f in os.listdir(SEPARATED_BURSTS_DIR) if f.endswith('.csv')]
    print(f"  Found {len(burst_files)} burst files: {burst_files[:5]}...")

Checking available data files...

Captured files: data/captured_files/
  Found 1 PCAP files: ['how_deep_is_the_indian_ocean_5_30s.pcap']

Trace CSV files: data/trace_csv_files/
  Found 1 trace files

Separated bursts files: data/seperated_bursts_files/
  Found 21 burst files: ['what_is_the_weather_4_bursts.csv', 'what_is_the_weather_3_bursts.csv', 'good_morning_1_bursts.csv', 'tell_me_a_joke_1_bursts.csv', 'good_morning_4_bursts.csv']...


In [16]:
# Generate synthetic sample data for testing (if no real data available)
def generate_synthetic_burst_data(output_dir=SEPARATED_BURSTS_DIR, n_queries=20, packets_per_query=50):
    """
    Generate synthetic burst traffic data for testing.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    query_templates = [
        'what_is_the_weather',
        'tell_me_a_joke',
        'play_music',
        'set_alarm',
        'good_morning'
    ]
    
    for i in range(n_queries):
        query_name = query_templates[i % len(query_templates)]
        filename = f"{query_name}_{i//len(query_templates)+1}_bursts.csv"
        filepath = os.path.join(output_dir, filename)
        
        # Generate random packet data with query-specific patterns
        np.random.seed(i)
        n_packets = packets_per_query + np.random.randint(-10, 10)
        
        # Different packet size distributions for different queries
        base_size = 100 + (i % len(query_templates)) * 50
        sizes = np.random.poisson(base_size, n_packets)
        
        # Time intervals
        times = np.cumsum(np.random.exponential(0.1, n_packets))
        
        # Directions (1 = outgoing, -1 = incoming)
        directions = np.random.choice([1, -1], n_packets)
        
        # Create DataFrame with new column names
        df = pd.DataFrame({
            'packet_id': range(n_packets),
            'timestamp': times,
            'length': sizes,
            'direction': directions,
            'burst_id': 1
        })
        
        # Add additional columns that separate_bursts produces
        df['gap_before'] = df['timestamp'].diff().fillna(float('inf'))
        df['gap_after'] = df['timestamp'].diff(periods=-1).abs().fillna(float('inf'))
        df['rel_time'] = df['timestamp'] - df['timestamp'].min()
        df['viz_bytes'] = df.apply(lambda x: x['length'] if x['direction'] == 1 else -x['length'], axis=1)
        df['cumulative_sum'] = df['viz_bytes'].cumsum()
        
        df.to_csv(filepath, index=False)
    
    print(f"Generated {n_queries} synthetic burst CSV files in {output_dir}")


# Check if we have burst data, if not generate synthetic data
burst_files = []
if os.path.exists(SEPARATED_BURSTS_DIR):
    burst_files = [f for f in os.listdir(SEPARATED_BURSTS_DIR) if f.endswith('.csv')]

if len(burst_files) < 10:
    print("Generating synthetic sample data...")
    generate_synthetic_burst_data()
    burst_files = [f for f in os.listdir(SEPARATED_BURSTS_DIR) if f.endswith('.csv')]
    print(f"Now have {len(burst_files)} burst files")

In [17]:
# Run the complete workflow on the sample data
print("Running complete workflow on sample data...")
print("=" * 60)

# Test with Bayes classifier
buflo_params = {'d': 600, 'f': 50, 't': 20}

results = run_complete_workflow(
    SEPARATED_BURSTS_DIR, 
    buflo_params=buflo_params,
    attack_method='bayes',
    n_folds=3
)

# Compare results
compare_results(results)

Running complete workflow on sample data...
Step 1: Loading original (undefended) data...
Loaded 21 samples with 21 classes
Feature dimension: 61

Step 2: Running attack on undefended traffic...
Filtering out these classes before cross-validation...


ValueError: min() iterable argument is empty

## Section 9: Testing with Different Methods

In [18]:
# Test with different attack methods
print("\n" + "=" * 60)
print("Testing with different classification methods")
print("=" * 60)

methods = ['bayes', 'svm']
all_results = {}

for method in methods:
    print(f"\n--- Testing {method.upper()} classifier ---")
    try:
        X, y, label_map = load_dataset_from_bursts(SEPARATED_BURSTS_DIR, feature_method=method)
        if len(X) > 0:
            avg_acc, avg_rank, _ = n_fold_cross_validation(X, y, n_folds=3, method=method)
            all_results[method] = {'accuracy': avg_acc, 'avg_rank': avg_rank}
    except Exception as e:
        print(f"Error with {method}: {e}")


# Summary
print("\n" + "=" * 60)
print("SUMMARY: Classification Method Comparison")
print("=" * 60)

for method, res in all_results.items():
    print(f"{method.upper()}: Accuracy = {res['accuracy']:.4f}")


Testing with different classification methods

--- Testing BAYES classifier ---
Loaded 21 samples with 21 classes
Feature dimension: 61
Filtering out these classes before cross-validation...
Error with bayes: min() iterable argument is empty

--- Testing SVM classifier ---
Error with svm: Unknown feature method: svm

SUMMARY: Classification Method Comparison


## Conclusion

This notebook provides a complete sequential workflow that combines:

1. **Data Preprocessing**: PCAP to CSV conversion using TrafficLoader
2. **Burst Separation**: Using separate_bursts function from services
3. **BuFLO Defense**: Application of the BuFLO countermeasure
4. **VCFP Attack**: Classification of voice commands from traffic
5. **Comparison Analysis**: Evaluation of defense effectiveness

The workflow demonstrates how BuFLO defense can reduce the accuracy of voice command fingerprinting attacks by obfuscating traffic patterns.

## Key Updates from Original Notebook:
- Uses service modules (TrafficLoader, separate_bursts) from the repository
- Updated directory paths to match repository structure
- Updated CSV column names (length instead of size, timestamp instead of time)
- Added burst_id handling for multi-burst PCAP files
- Removed deprecated sklearn.externals.joblib import